# Quick demo: score incoming 311 cases with the trained model

Loads `artifacts/model.joblib` and scores cases straight away. No data download, no training,
under a minute on a fresh Colab runtime. The full reproducible pipeline, from the live API
download to the test-set evaluation, is `demo.ipynb`, and it takes 10 to 15 minutes because
of the download.

"SLA" here means a service window derived from how long each request type historically took,
a proxy rather than NYC's official SLA: the city's own `due_date` field is populated for 0.53%
of cases.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Fakhrillo/nyc311-sla-breach-prediction.git"

if "google.colab" in sys.modules and not Path("src").exists():
    # Same restart-safe setup as demo.ipynb: "Restart session" keeps the clone but
    # resets the working directory, so only clone when it is not there yet.
    if not Path("repo").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "repo"], check=True)
    os.chdir("repo")
    # Only what unpickling the model needs. The saved pipeline was fitted with
    # scikit-learn 1.9 and will not load on an older one.
    subprocess.run(["pip", "install", "-q", "scikit-learn>=1.9"], check=False)

sys.path.insert(0, str(Path.cwd() / "src"))
import sla as S

## The shipped model

What `train.py` saved: the fitted pipeline, the service windows it was trained against, the
expediting threshold, and the test-set report from the run that produced it.

In [ ]:
bundle = S.load_model()
test = bundle["metrics"]["test"]

print(f"model:      {bundle['metrics']['final_model']} {bundle['metrics']['final_params']}")
print(f"threshold:  {bundle['threshold']:.3f}  (top 20% of validation scores)")
print(f"test set:   {test['n']:,} unseen cases, breach rate {test['positive_rate']:.3f}")
print(f"            PR-AUC {test['pr_auc']:.4f} | recall at 20% budget {test['recall_at_20pct']:.4f}")

## Score four incoming cases

Each record is what a call taker has at intake. Change any field and re-run the cell to score
a different case: a later hour, a heavier backlog, another borough.

In [ ]:
import pandas as pd

cases = {
    "noise call, quiet queue": dict(created_date="2024-03-20T23:10:00",
        complaint_type="Noise - Residential", agency="NYPD", borough="BROOKLYN",
        open_data_channel_type="PHONE", agency_backlog=200),
    "heat complaint, heavy backlog": dict(created_date="2024-03-20T08:00:00",
        complaint_type="HEAT/HOT WATER", agency="HPD", borough="BRONX",
        open_data_channel_type="ONLINE", agency_backlog=9000, agency_7d_volume=15000),
    "water leak, weekend 3am": dict(created_date="2024-03-23T03:00:00",
        complaint_type="WATER LEAK", agency="HPD", borough="MANHATTAN",
        open_data_channel_type="MOBILE", agency_backlog=5000),
    "request type never seen in training": dict(created_date="2024-03-20T12:00:00",
        complaint_type="Rodent Sighting", agency="DOHMH", borough="QUEENS"),
}

pd.DataFrame({name: S.predict_one(rec, bundle) for name, rec in cases.items()}).T.drop(columns="threshold")

## Bad input is rejected, not guessed at

A garbled record that quietly returns a probability is worse than one that errors, so these
raise `ValueError` with a message safe to show a user.

In [ ]:
bad = [
    ({"complaint_type": "Noise"}, "missing required fields"),
    (dict(created_date="not-a-date", complaint_type="X", agency="A", borough="B"), "bad timestamp"),
    (dict(created_date="2024-03-20", complaint_type="X", agency="A", borough="B",
          agency_backlog=-1), "negative backlog"),
    (dict(created_date="2024-03-20", complaint_type="  ", agency="A", borough="B"), "blank type"),
]
for rec, why in bad:
    try:
        S.predict_one(rec, bundle)
        print(f"NOT CAUGHT: {why}")
    except ValueError as e:
        print(f"rejected ({why}): {e}")